# Optimized Baseline: Prefix-Cache and Load-Aware Routing

This notebook walks through the **optimized baseline** well-lit path in detail.
The optimized baseline is llm-d's recommended starting configuration for production.
It combines two routing signals in the Endpoint Picker (EPP):

1. **Prefix-cache affinity** - Route requests to replicas that already have the
   prompt prefix in GPU KV-cache memory.
2. **Load awareness** - Avoid overloaded replicas by considering queue depth and
   KV-cache utilization.

Together, these provide up to **3x throughput** and **2x TTFT improvement** compared
to round-robin load balancing.

## Architecture

```
                    +------------------+
                    |   Client / App   |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Envoy Gateway   |  Port 8080 - OpenAI-compatible API
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |   EPP Router     |  Scores each replica:
                    |                  |    prefix_cache_score * w1
                    |                  |    + load_score * w2
                    +--------+---------+
                             |
              +--------------+--------------+
              |              |              |
              v              v              v
        +-----------+  +-----------+  +-----------+
        | vLLM      |  | vLLM      |  | vLLM      |
        | Replica 0 |  | Replica 1 |  | Replica 2 |
        | (GPU)     |  | (GPU)     |  | (GPU)     |
        +-----------+  +-----------+  +-----------+
```

Each vLLM replica exposes metrics about its prefix cache contents and current load.
The EPP scrapes these metrics and uses them to pick the best replica for each request.

In [ ]:
# Set environment variables for this guide
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["GUIDE_NAME"] = "optimized-baseline"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"
os.environ["REPLICAS"] = "3"
os.environ["GUIDE_PATH"] = "llm-d-production-stack/guides/optimized-baseline"

print(f"Guide: {os.environ['GUIDE_NAME']}")
print(f"Model: {os.environ['MODEL_NAME']}")
print(f"Replicas: {os.environ['REPLICAS']}")

In [ ]:
# Deploy the model server using kustomize
# This creates a Deployment with the specified number of replicas,
# each requesting GPU resources and running vLLM with prefix caching enabled.

!kubectl apply -k $GUIDE_PATH/model-server/ -n $NAMESPACE

print("\nModel server deployment applied.")
print("Waiting for model server pods to be ready...")

!kubectl rollout status deployment/vllm -n $NAMESPACE --timeout=600s

print("Model server is ready.")

In [ ]:
# Deploy the router (EPP + Envoy Gateway)
# The router kustomization includes:
#   - EPP Deployment and Service
#   - InferencePool resource defining the backend model servers
#   - Gateway and HTTPRoute resources for Envoy

!kubectl apply -k $GUIDE_PATH/router/ -n $NAMESPACE

print("\nRouter deployment applied.")
print("Waiting for EPP pods to be ready...")

!kubectl rollout status deployment/epp -n $NAMESPACE --timeout=120s

print("Router is ready.")

In [ ]:
# Verify the full deployment
print("=== Pods ===")
!kubectl get pods -n $NAMESPACE -o wide

print("\n=== Services ===")
!kubectl get svc -n $NAMESPACE

print("\n=== InferencePool ===")
!kubectl get inferencepool -n $NAMESPACE -o yaml

print("\n=== Gateway ===")
!kubectl get gateway -n $NAMESPACE

## EPP Configuration: The Scoring Pipeline

The EPP uses a **scoring pipeline** to rank replicas for each incoming request.
The pipeline runs a series of scorers, each producing a score between 0 and 1.
These scores are combined with configurable weights to produce a final score.

The optimized baseline uses two scorers:

### Prefix Cache Scorer
- Hashes the prompt token blocks and compares them against each replica's
  reported cached blocks.
- Score = (number of matched prefix blocks) / (total prefix blocks in request).
- A score of 1.0 means the entire prompt prefix is already cached on that replica.

### Load Scorer
- Queries each replica's current queue depth and KV-cache utilization percentage.
- Score = 1.0 - (normalized_load). A fully idle replica scores 1.0; a fully
  loaded replica scores 0.0.

### Final Score
```
final_score = prefix_cache_score * 0.7 + load_score * 0.3
```

The replica with the highest final score is selected to serve the request.

In [ ]:
# Show the EPP config being used in this deployment
# This is the InferencePool spec that configures the scoring pipeline

!kubectl get configmap epp-config -n $NAMESPACE -o yaml

print("\n--- Key configuration fields ---")
print("")
print("The scoring pipeline is defined in the EPP configuration.")
print("Default scorer weights for the optimized baseline:")
print("  prefix-cache-scorer: weight = 0.7")
print("  load-aware-scorer:   weight = 0.3")
print("")
print("You can adjust these weights by editing the ConfigMap and restarting the EPP.")

In [ ]:
# Run a benchmark comparing prefix-cache routing vs round-robin
# We use a synthetic workload with shared prefixes to highlight the difference.

import subprocess
import json

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Gateway endpoint: {GATEWAY_IP}:8080")
print()
print("Running benchmark with shared-prefix workload...")
print("This simulates a multi-tenant scenario where many requests share")
print("the same system prompt and context prefix.")
print()

# Using a simple benchmark script with curl in a loop
# In production, you would use tools like genai-perf or llmperf
!python3 -c "
import time, subprocess, json, statistics

system_prompt = 'You are a helpful customer service agent for Acme Corp. ' * 50  # ~500 tokens shared prefix
questions = [
    'What is your return policy?',
    'How do I track my order?',
    'Can I change my shipping address?',
    'What payment methods do you accept?',
    'How long does shipping take?',
]

latencies = []
for q in questions:
    payload = json.dumps({
        'model': 'Qwen/Qwen3-32B',
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': q},
        ],
        'max_tokens': 50,
    })
    start = time.time()
    result = subprocess.run(
        ['curl', '-s', f'http://{GATEWAY_IP}:8080/v1/chat/completions',
         '-H', 'Content-Type: application/json', '-d', payload],
        capture_output=True, text=True
    )
    elapsed = time.time() - start
    latencies.append(elapsed)
    print(f'  Request {len(latencies)}: {elapsed:.3f}s')

print()
print(f'Median latency: {statistics.median(latencies):.3f}s')
print(f'P99 latency:    {sorted(latencies)[int(len(latencies)*0.99)]:.3f}s')
print()
print('With prefix-cache routing, requests 2-5 should be significantly faster')
print('than request 1, because the shared system prompt is already cached.')
"

## Expected Results

In benchmarks with shared-prefix workloads, the optimized baseline typically shows:

| Metric | Round-Robin | Prefix-Cache + Load-Aware |
|--------|------------|---------------------------|
| **Throughput** | 1x (baseline) | ~3x |
| **TTFT (median)** | 1x (baseline) | ~0.5x (2x improvement) |
| **TTFT (P99)** | 1x (baseline) | ~0.3x (3x improvement) |
| **Cache hit rate** | ~10-15% (random) | ~80-95% |

The improvement is most dramatic when:
- Many requests share a common prefix (system prompts, few-shot examples)
- The prefix is long (hundreds or thousands of tokens)
- You have multiple replicas (more routing choices = more benefit)

For workloads with unique prompts and no shared prefix, the load-aware component
still provides better tail latency than round-robin.

In [ ]:
# Monitor prefix cache hit rate in real time
# The EPP exposes Prometheus metrics that you can query

import time
import urllib.request

EPP_METRICS = "http://localhost:9090/metrics"  # Port-forward: kubectl port-forward svc/epp -n llm-d 9090:9090

print("Monitoring prefix cache hit rate (Ctrl+C to stop)...")
print("Tip: Run this while sending requests in another terminal to see the cache warm up.")
print()

try:
    for i in range(10):  # Monitor for 10 iterations
        try:
            with urllib.request.urlopen(EPP_METRICS, timeout=2) as resp:
                metrics = resp.read().decode("utf-8")

            for line in metrics.split("\n"):
                if line.startswith("epp_prefix_cache_hit_ratio"):
                    print(f"[{time.strftime('%H:%M:%S')}] {line}")
                elif line.startswith("epp_request_total"):
                    print(f"[{time.strftime('%H:%M:%S')}] {line}")
        except Exception as e:
            print(f"[{time.strftime('%H:%M:%S')}] Could not reach metrics: {e}")

        time.sleep(2)

except KeyboardInterrupt:
    print("\nStopped monitoring.")

## Tuning Tips

### Scorer Weights

Adjust the ratio between prefix-cache and load scoring based on your workload:

- **High prefix sharing** (chatbots with fixed system prompts): Increase
  prefix-cache weight (e.g., 0.8/0.2).
- **Diverse prompts** (code generation, varied tasks): Increase load weight
  (e.g., 0.5/0.5) since cache hits will be rare.
- **Bursty traffic**: Increase load weight to prevent hotspots during spikes.

### Block Size

vLLM's prefix cache operates in blocks (default 16 tokens). Prompts that are
not multiples of the block size will have partial last blocks that cannot be
cached. If your prompts have very regular structure, consider padding to
block boundaries.

### Number of Replicas

More replicas give the EPP more routing choices, but also spread the cache
across more servers. For prefix-heavy workloads, there is a sweet spot where
adding more replicas dilutes the cache. Start with 2-4 replicas and benchmark
to find the optimal count for your workload.